[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/02_extract.ipynb)

# Extract — kinds, checkpoints, quote support

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Address begins when we ask a model for structured points. Multilogue keeps
extraction generic: a kind name, a prompt template, a sentinel for “nothing
here”, and an optional quote-support gate so each phrase is grounded in the
speaker's words. Checkpoints make reruns resume without paying twice. This
notebook teaches those ideas on the public CIMT corpus; offline pedagogy uses
`phrases_from_units` so we never invent civic text.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

Extraction is the first Address stage. The library does not hard-code
“concern” or “benefit”: those are *kinds* supplied by the consumer. What
multilogue owns is the machinery — parse items from a model response, verify
quotes against unit text, and persist results under a fingerprint of prompt +
model so the same work is not repeated. When a live model is unavailable,
`phrases_from_units` builds a phrase table from Access units (one phrase per
unit; kind from gold `meta.code` when present).


## Checkpoints

Model stages run through `StageRunner`: append-only JSONL keyed by item, with a
fingerprint of prompt + model. Reruns resume; fingerprint changes supersede;
replay needs no API. That is engineering for reproducible dialogue analysis, not
a modelling claim.


## Quote support

A phrase without a recoverable quote is easy for a model to invent. The score
`quote_support` asks how well the quoted span sits in the unit text. Write $q$
for the quote and $s$ for the source unit. Support is high when a normalised
form of $q$ appears (approximately) in $s$. Thresholds are a design choice —
high enough to catch fabrications, low enough to tolerate light paraphrase.


In [ ]:
from IPython.display import display
from multilogue.text import quote_support
from multilogue import viz


In [ ]:
import os
os.environ.setdefault("MULTILOGUE_ACCESS_YES", "1")
from multilogue.experiment import load_cimt_format_pair
from multilogue.extract import phrases_from_units

units, cimt_meta = load_cimt_format_pair(max_units_per=40)
if units is None:
    raise RuntimeError(
        "CIMT Access failed: "
        + str(cimt_meta.get("error", "unknown"))
        + ". Need network once to populate MULTILOGUE_DATAPATH / ~/.cache/multilogue."
    )
print(cimt_meta)
phrases = phrases_from_units(units, extra_fields=["group", "format"])
print(f"{len(units)} units → {len(phrases)} phrases (unit-as-phrase; LLM extract optional)")


In [ ]:
display(phrases.head())
yd = (
    phrases.groupby(["format", "kind"], dropna=False)
    .size()
    .rename("kept")
    .reset_index()
)
display(yd)
viz.yield_bars(yd.rename(columns={"format": "group"}), path=FIG / "02_yield.png")


Yield counts show how many unit-as-phrases of each kind appear by format.
With a live client the same columns come from `phrase_table` after
`StageRunner` extract; the Access corpus stays public either way.


In [ ]:
demo_text = units[0]["text"]
span = " ".join(demo_text.split()[:8])
print("support(exact span):", round(quote_support(span, demo_text), 3))
print("support(noise):", round(quote_support("completely unrelated words", demo_text), 3))
if "verified" in phrases.columns:
    display(phrases.groupby(["format", "kind"]).verified.mean().rename("pct_verified"))


# Outlook

Phrases alone are not a comparison. Notebook 03 places them in a shared
embedding space with soft MPPCA topics (harden only for labels / lenses).
